In [113]:
import pandas as pd
import numpy as np

import re
import string

from sklearn.preprocessing import LabelEncoder


import joblib

In [114]:
df = pd.read_csv('../data/enron_spam_data.csv')

In [115]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33716 entries, 0 to 33715
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Message ID  33716 non-null  int64 
 1   Subject     33427 non-null  object
 2   Message     33345 non-null  object
 3   Spam/Ham    33716 non-null  object
 4   Date        33716 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.3+ MB


In [116]:
df.sample(10)

,Message ID,Subject,Message,Spam/Ham,Date
8380,8380,re : enroll in intro to java at productivity p...,marty :\nhow many do we need to enroll in orde...,ham,2001-01-26
6674,6674,var update memo,"hi grant ,\nas requested i have sent a quick s...",ham,2000-08-01
33593,33593,"john digweed charset = windows - 1252 "" >",jonh\ndigweed dj vibe\nsábado 23 de julho @\np...,spam,2005-07-19
23472,23472,action from last doorstep meeting,ted / sally\nwe took as an action from the las...,ham,2000-10-31
2023,2023,revison # 1 - enron / hpl actuals for november...,"november 10 , 2000\nteco tap 40 . 000 / enron ...",ham,2000-11-13
17014,17014,real time meeting / questions,two things -\n1 . weather tracking - did we de...,ham,2001-08-06
14124,14124,enron mentions - 11 / 26 / 01,dynegy seems to have options in enron deal\nth...,ham,2001-11-26
290,290,curtains,hey ! the curtains look good - - i hope they d...,ham,2000-02-10
9151,9151,re : pending approval for ibuyit request for w...,"kaminski : eva : remedy 412144\nvince ,\nwelco...",ham,2001-04-20
30253,30253,you can save up to 70 % on cialfy,cialis ( super viagra ) at a 2 . 75 per dose\n...,spam,2004-11-03


In [117]:
df.shape

(33716, 5)

In [118]:
df = df.drop(columns=["Message ID", "Date"])

In [119]:
df.head()

,Subject,Message,Spam/Ham
0,christmas tree farm pictures,NaN,ham
1,"vastar resources , inc .","gary , production from the high island larger ...",ham
2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham
3,re : issue,fyi - see note below - already done .\nstella\...,ham
4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham


In [120]:
df.isnull().sum()

Subject     289
Message     371
Spam/Ham      0
dtype: int64

In [121]:
df["Subject"] = df["Subject"].fillna("")
df["Message"] = df["Message"].fillna("")

In [122]:
df.isnull().sum()

Subject     0
Message     0
Spam/Ham    0
dtype: int64

In [123]:
label_encoder = LabelEncoder()

df["Spam/Ham"] = label_encoder.fit_transform(df["Spam/Ham"])

In [124]:
df.head()

,Subject,Message,Spam/Ham
0,christmas tree farm pictures,,0
1,"vastar resources , inc .","gary , production from the high island larger ...",0
2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,0
3,re : issue,fyi - see note below - already done .\nstella\...,0
4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,0


In [125]:
joblib.dump(label_encoder, "label_encoder.pkl")

['label_encoder.pkl']

In [126]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"<.*?>", "", text)

    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [127]:
df["Subject"] = df["Subject"].apply(clean_text)

df["Message"] = df["Message"].apply(clean_text)

In [128]:
df.head()

,Subject,Message,Spam/Ham
0,christmas tree farm pictures,,0
1,vastar resources inc,gary production from the high island larger bl...,0
2,calpine daily gas nomination,calpine daily gas nomination 1 doc,0
3,re issue,fyi see note below already done stella forward...,0
4,meter 7268 nov allocation,fyi forwarded by lauri a allen hou ect on 12 1...,0


In [129]:
df["combined_text"] = (
    df["Subject"] + " " + df["Message"]
)

In [130]:
df["combined_text"].head()

0                        christmas tree farm pictures 
1    vastar resources inc gary production from the ...
2    calpine daily gas nomination calpine daily gas...
3    re issue fyi see note below already done stell...
4    meter 7268 nov allocation fyi forwarded by lau...
Name: combined_text, dtype: object

In [131]:
df = df.drop(columns=["Subject", "Message"])

In [132]:
df[0:2]

,Spam/Ham,combined_text
0,0,christmas tree farm pictures
1,0,vastar resources inc gary production from the ...


In [134]:
import re
from urllib.parse import urlparse

def extract_url_features(text):
    """
    Extract meaningful information from URLs and append it to the text.
    """

    extracted_tokens = []

    pattern = r'https?://[^\s]+'

    urls = re.findall(pattern, text)

    for url in urls:

        parsed = urlparse(url)

        # protocol
        if parsed.scheme:
            extracted_tokens.append(parsed.scheme.lower())

        # domain + tld
        domain_parts = parsed.netloc.lower().replace("www.", "").split(".")

        if len(domain_parts) >= 2:
            extracted_tokens.append(domain_parts[-2])   # domain
            extracted_tokens.append(domain_parts[-1])   # tld
        elif len(domain_parts) == 1:
            extracted_tokens.append(domain_parts[0])

        # query exists
        if parsed.query:
            extracted_tokens.append("query")

    # Remove original URLs
    cleaned_text = re.sub(pattern, "", text)

    # Append extracted values
    if extracted_tokens:
        cleaned_text += " " + " ".join(extracted_tokens)

    return " ".join(cleaned_text.split())

In [135]:
df['combined_text'] = df['combined_text'].apply(extract_url_features)

In [136]:
df.sample(5)

,Spam/Ham,combined_text
6406,0,control and echelon very interesting websites ...
13384,0,fw 2002 headcount changes laynie per my swappe...
17955,0,start date 1 31 02 hourahead hour 17 start dat...
10049,1,our cool medz hello welcome to medzonli decapi...
24632,1,printer ink cartridges refill kits from  4 85...


In [137]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33716 entries, 0 to 33715
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Spam/Ham       33716 non-null  int64 
 1   combined_text  33716 non-null  object
dtypes: int64(1), object(1)
memory usage: 526.9+ KB


In [ ]:
df.to_csv("../data/processed_data.csv", index=False)